# Setup

In [ ]:
# load required libraries with requirements.txt
! pip install -r requirements.txt

In [3]:
#Initial imports

import matplotlib.pyplot as plt
import mysql.connector as mysql
import pandas as pd
import numpy as np
import json

from DbConnector import DbConnector

In [ ]:
# Import datasets
credits_df = pd.read_csv("movies_ds/credits.csv")
keywords_df = pd.read_csv("movies_ds/keywords.csv")
links_small_df = pd.read_csv("movies_ds/links_small.csv")
links_df = pd.read_csv("movies_ds/links.csv")
movies_metadata_df = pd.read_csv("movies_ds/movies_metadata.csv")
ratings_small_df = pd.read_csv("movies_ds/ratings_small.csv")
ratings_df = pd.read_csv("movies_ds/ratings.csv")

In [ ]:
# Cleaning of the data
credits_df_clean = credits_df.copy()
keywords_df_clean = keywords_df.copy()
links_small_df_clean = links_small_df.copy()
links_df_clean = links_df.copy()
movies_metadata_df_clean = movies_metadata_df.copy()
ratings_small_df_clean = ratings_small_df.copy()
ratings_df_clean = ratings_df.copy()

# Getting rid of duplicates (movies with the same id)
movies_metadata_df_clean = movies_metadata_df_clean.drop_duplicates(subset='id')

# Getting rid of movies with unknown release date
movies_metadata_df_clean = movies_metadata_df_clean.dropna(axis=0, subset='release_date')

In [ ]:
from datetime import datetime

def timestamp_to_datetime(timestamp):
    """
    In: timestamp (number of seconds since January 1st, 1970)
        typically items from the 'timestamp' column in the ratings_df dataset are in this format
    Out: datetime following this format: YYYY-MM-DD HH:MM:SS
    """
    return datetime.fromtimestamp(timestamp)

def date_to_datetime(date):
    """
    In: date following this format: YYYY-MM-DD as a string
        typically items from the 'release_date' column in the movies_metadata_df dataset are in this format
    Out: datetime following this format: YYYY-MM-DD HH:MM:SS
    """
    return datetime.strptime(date, "%Y-%m-%d")

# Getting rid of reviews posted before the movie release date
# Only doable for ratings_small_df, takes too long to compute for ratings_df
nb_ratings_deleted = 0
print(len(ratings_small_df_clean))
i = 0
for index, row in ratings_small_df_clean.iterrows():
    rating_datetime = timestamp_to_datetime(row['timestamp'])
    movie_id = str(int(row['movieId']))
    movie = movies_metadata_df_clean.loc[movies_metadata_df_clean['id'] == movie_id]
    if not movie.empty:
        movie = movie.iloc[0]
        movie_release_datetime = date_to_datetime(str(movie['release_date']))
        if rating_datetime < movie_release_datetime:
            ratings_small_df_clean.drop(index)
            nb_ratings_deleted += 1
    i += 1
    print(i)
print(str(nb_ratings_deleted) + " ratings deleted")

In [ ]:
from DbConnector import DbConnector
import pymongo

class DbConnector:
    """
    Connects to the MongoDB server on the Ubuntu virtual machine.
    Connector needs HOST, USER and PASSWORD to connect.

    Example:
    HOST = "tdt4225-00.idi.ntnu.no" // Your server IP address/domain name
    USER = "testuser" // This is the user you created and added privileges for
    PASSWORD = "test123" // The password you set for said user
    """

    def __init__(self,
                 DATABASE='DATABASE_NAME',
                 HOST="tdt4225-03.idi.ntnu.no",
                 USER="TEST_USER",
                 PASSWORD="test123"):
        uri = "mongodb://%s:%s@%s:%d/%s?authSource=%s" % (USER, PASSWORD, HOST, PORT, DATABASE, DATABASE)
        # Connect to the databases
        try:
            self.client = MongoClient(uri)
            self.db = self.client[DATABASE]
        except Exception as e:
            print("ERROR: Failed to connect to db:", e)

        # get database information
        print("You are connected to the database:", self.db.name)
        print("-----------------------------------------------\n")

    def close_connection(self):
        # close the cursor
        # close the DB connection
        self.client.close()
        print("\n-----------------------------------------------")
        print("Connection to %s-db is closed" % self.db.name)

class ExampleProgram:

    def __init__(self):
        self.connection = DbConnector()
        self.client = self.connection.client
        self.db = self.connection.db

    def create_coll(self, collection_name):
        collection = self.db.create_collection(collection_name)    
        print('Created collection: ', collection)

    def insert_documents(self, collection_name):
        docs = [
            {
                "_id": 1,
                "name": "Bobby",
                "courses": 
                    [
                    {'code':'TDT4225', 'name': ' Very Large, Distributed Data Volumes'},
                    {'code':'BOI1001', 'name': ' How to become a boi or boierinnaa'}
                    ] 
            },
            {
                "_id": 2,
                "name": "Bobby",
                "courses": 
                    [
                    {'code':'TDT02', 'name': ' Advanced, Distributed Systems'},
                    ] 
            },
            {
                "_id": 3,
                "name": "Bobby",
            }
        ]  
        collection = self.db[collection_name]
        collection.insert_many(docs)
        
    def fetch_documents(self, collection_name):
        collection = self.db[collection_name]
        documents = collection.find({})
        for doc in documents: 
            pprint(doc)
        

    def drop_coll(self, collection_name):
        collection = self.db[collection_name]
        collection.drop()

        
    def show_coll(self):
        collections = self.client['test'].list_collection_names()
        print(collections)

    def exercise_5(self):
        query = db.collection('movies_metadata').aggregate(
            [
                { $group: {
                        _id: { 
                            $dateToString: { format: "%d-%m-%Y %h:%m:%s", decade: "$YY00" },
                            $primaryGenre: 
                        },

                    } 
                }
            ])
 
        self.cursor.execute(query)
        midnight_crossers = self.cursor.fetchall()
        print(tabulate(midnight_crossers[:10], headers=self.cursor.column_names))


def main():
    program = None
    try:
        program = Program()

        # program.initiate_db(df_clean)  WILL RESET THE DATABASE

        # program.check_data()
        # program.show_tables()

        # program.exercise_1()
        
    except Exception as e:
        print("ERROR: Failed to use database:", e)
    finally:
        if program:
            program.connection.close_connection()

if __name__ == '__main__':
    main()
    print("Done")